In [ ]:
from pathlib import Path
import pathlib
from typing import Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

In [ ]:
path = "../../"
path = Path(path).expanduser()
import sys
sys.path.insert(0, str(path))

In [ ]:
import decode
import decode.neuralfitter.inference.functional as infer_func
print(decode.__file__)
log = decode.generic.logging.get_logger(__name__)

%config InlineBackend.figure_format='retina'

In [ ]:
import torch
import torch.nn.functional as F

def add_background_noise(
    frames,
    mode="perlin",
    amplitude=20,
    sigma=20,
):
    frames = frames.clone()

    if mode == "gaussian":
        bg = amplitude * torch.randn_like(frames)

    elif mode == "perlin":

        # low-frequency random field
        bg = torch.randn_like(frames)

        k = int(max(3, sigma))
        if k % 2 == 0:
            k += 1

        ax = torch.arange(k, device=frames.device) - k // 2
        kernel = torch.exp(-0.5 * (ax / sigma) ** 2)
        kernel /= kernel.sum()

        kernel2d = kernel[:, None] * kernel[None, :]
        kernel2d = kernel2d[None, None]

        bg = F.conv2d(
            bg[:, None],
            kernel2d,
            padding=k // 2,
        )[:, 0]

        bg = bg - bg.min()
        bg = bg / bg.max()

        bg = amplitude * bg

    else:
        raise ValueError(mode)

    return frames + bg

def gaussian_blur(frames, kernel_size=5, sigma=1.0):
    ax = torch.arange(kernel_size, device=frames.device) - kernel_size // 2
    kernel = torch.exp(-0.5 * (ax / sigma) ** 2)
    kernel = kernel / kernel.sum()
    kernel2d = kernel[:, None] * kernel[None, :]
    kernel2d = kernel2d[None, None]

    x = frames[:, None]  # [N,1,H,W]
    x = F.conv2d(x, kernel2d, padding=kernel_size // 2)
    return x[:, 0]

def shift_channel2(frames, shift_x=0, shift_y=0):
    frames = frames.clone()

    ch2 = frames[:, 64:, :].clone()

    # shift
    ch2 = torch.roll(ch2, shifts=(shift_y, shift_x), dims=(1, 2))

    # zero padding instead of wrap-around
    if shift_y > 0:
        ch2[:, :shift_y, :] = 0
    elif shift_y < 0:
        ch2[:, shift_y:, :] = 0

    if shift_x > 0:
        ch2[:, :, :shift_x] = 0
    elif shift_x < 0:
        ch2[:, :, shift_x:] = 0

    frames[:, 64:, :] = ch2

    return frames


def rot_90(frames):
    ch1 = frames[:, :64, :]
    ch2 = frames[:, 64:, :]

    ch1 = torch.rot90(ch1, k=1, dims=(1,2))
    ch2 = torch.rot90(ch2, k=1, dims=(1,2))

    frames = torch.cat([ch1, ch2], dim=1)
    
    return frames

In [ ]:
path_frames = "../../data/Fig1e-Challenge/"

path_frames = Path(path_frames).expanduser()

path_training = "../../outputs/Fig1e-challenge-2026-05-27_15-24-26-229646"
path_training = Path(path_training).expanduser()
path_cfg = path_training / "param_run.yaml"
path_ckpt = next((path_training).glob("*.ckpt"))

model_identifier = f"model_{path_ckpt.parent.stem}"

cfg = decode.io.param.load(path_cfg)
gain_correct = True

device = ["cuda:0"]

for s in cfg["Hardware"]["device"]:
    cfg["Hardware"]["device"][s] = device[0]

cfg["Camera"][0]["specs"]["flip"]["channel"] = -2
cfg["Camera"][1]["specs"]["flip"]["channel"] = -2

path_subs = [sorted(path_frames.glob("*.tif"))[0].parent]
mode_camera = "rois"

In [ ]:

from decode.io.frames import TiffTensor 
import math
for p in path_subs:
    pframe = sorted(p.glob("*.tif"))[0]
    frames = decode.io.frames.load_tif(pframe, auto_ome=True, memmap=True)
    print(f"Paths: \n{pframe}\nsize: {frames.size()}")
    frames = frames[:].clone() + 50
    frames[:,64:,:] = frames[:,64:,:].flip(1) + 100
    
    frames = gaussian_blur(frames, sigma=1.0)
    frames = add_background_noise(
                frames,
                mode="perlin",
                amplitude=50,
                sigma=20,
            )
    
    
    frame_size = list(frames.size())
    frame_crop = [math.floor(frame_size[-2] /2 / 8) * 8, math.floor(frame_size[-1] / 8) * 8]
    print(f"Frame size: {frame_size} -> Crop: {frame_crop}")

    path_ckpt = next((path_training).glob("*.ckpt"))
    em_out, logger = infer_func.infer(
        frames,
        frame_crop=frame_crop,
        cfg=cfg,
        model=path_ckpt,
        mode="multi",
        trafo= None, # path_trafo,
        mode_camera=mode_camera,
        roi_shift=None, # shift[:2].tolist(),
        logger="debug",
        batch_size=16,
        num_workers=0,
        device = device,
    )

    em_wrong_psf = em_out.clone()
    # em_save.xyz_px += shift
    mask = (
        (em_wrong_psf.xyz[:, 0] < 58)
        & (em_wrong_psf.xyz[:, 0] > 8)
        & (em_wrong_psf.xyz[:, 1] < 58)
        & (em_wrong_psf.xyz[:, 1] > 8)
    )
    em_wrong_psf = em_wrong_psf[mask]
    em_wrong_psf = em_wrong_psf[em_wrong_psf.phot > 300]

    path_out = "../../results/" + f"{pframe.stem}_decode_plex_fit_{model_identifier}_gridding.h5"
    em_wrong_psf.save(path_out)
    print(f"Saved to {path_out}")
    break

In [ ]:
offsets = em_wrong_psf.xyz_px[:, :2].clone()
ix = torch.zeros_like(offsets, dtype=torch.long)

for i in range(2):
    offsets[:, i], ix[:, i] = decode.evaluation.predict_dist.px_pointer_dist(offsets[:, i], -0.5, 1., return_ix=True)
    
    
debias = decode.neuralfitter.de_bias.UniformizeOffsetCoordinateBased(10)
debias_by_z = decode.neuralfitter.de_bias.DebiasLateral(debias, 10)

# doffset = debias.forward(offsets, em.xyz_sig_px[:, :2])
doffset, ixx = debias_by_z.forward(offsets, em_wrong_psf.xyz_sig_px[:, :2], em_wrong_psf.xyz_px[:, 2])
_, ix_reverse = torch.sort(ixx, dim=0)

doffset = doffset[ix_reverse]


# new coordinates
em_debias_psf = em_wrong_psf.clone()
em_debias_psf.xyz_px[:, :2] = ix + doffset

path_out = "../../results/" + f"{pframe.stem}_decode_plex_fit_{model_identifier}_gridding_debias.h5"
em_debias_psf.save(path_out)
print(f"Saved to {path_out}")

In [ ]:
def load_csv(
    path: str | pathlib.Path,
    shift_frame_ix: int = 0,
) -> Tuple[dict, dict, dict]:
    """
    Load emitter data from csv file.

    Expected columns:
        gt
        frame
        xnano
        ynano
        znano
        intensity

    Args:
        path: csv file path
        shift_frame_ix: frame index shift

    Returns:
        (emitter_dict, emitter_meta, decode_meta)
    """

    df = pd.read_csv(path)

    emitter_dict = {
        "xyz": torch.tensor(
            df[["xnano", "ynano", "znano"]].values,
            dtype=torch.float32,
        ),
        "phot": torch.tensor(
            df["intensity "].values,
            dtype=torch.float32,
        ),
        "frame_ix": torch.tensor(
            df["frame"].values,
            dtype=torch.long,
        ) + shift_frame_ix,
    }

    return emitter_dict, {"xy_unit": "nm"}, {}

In [ ]:
gt_csv_path = "../../results/FigS2-gridding/activations.csv"
data, *_ = load_csv(gt_csv_path, -1)
px_size = (100., 100.)

em_gt = decode.EmitterSet(
        **data,
        prob=[1.] * len(data["frame_ix"]),
        xy_unit="nm",
        px_size=px_size,
    )

# em_gt.xyz_nm = em_gt.xyz_nm[:, [1, 0, 2]]
em_gt.xyz_nm[:, 1] = -em_gt.xyz_nm[:, 1] + 6400
em_gt.xyz_nm[:, 2] = em_gt.xyz_nm[:, 2] * -1

path_out = "../../results/" + f"{pframe.stem}_decode_plex_fit_{model_identifier}_gt.h5"
em_gt.save(path_out)

In [ ]:
em_path_true_psf = '../../results/FigS2-gridding/Fig1e-sequence-MT0.N2.HD-BP-Exp-as-stack_decode_plex_fit_model_2026-05-27_15-24-26-229646.h5'
# em_path = '/home/shah/projects/DECODE_PLEX/data/dual_color/240802_NC_ER_MT_TritonX100_08percent_1_MMStack_Default.ome_decode_plex_fit_model_2025-05-15_16-uiPSF-uiTrafo.h5'
em_true_psf = decode.EmitterSet.load(em_path_true_psf)

# em_path_wrong_psf = '../../results/sequence-MT0.N2.HD-BP-Exp-as-stack_decode_plex_fit_model_Fig1e-challenge-2026-05-27_15-24-26-229646.h5'
# em_wrong_psf = decode.EmitterSet.load(em_path_wrong_psf)


# em_path_debias_psf = '../../results/sequence-MT0.N2.HD-BP-Exp-as-stack_decode_plex_fit_model_Fig1e-challenge-2026-05-27_15-24-26-229646_debiased.h5'
# em_debias_psf = decode.EmitterSet.load(em_path_debias_psf)

path_out = '../../results/gridding/'
path_out = Path(path_out).expanduser()
path_out.mkdir(exist_ok=True, parents=True)

In [ ]:
matcher = decode.evaluation.match_emittersets.GreedyHungarianMatching(
    match_dims=2, dist_ax=None, dist_lat=250.
)
evaluator = decode.evaluation.evaluation.SMLMEvaluation()

In [ ]:
# calc
em_p_wrong = em_wrong_psf.clone()
em_p_debias = em_debias_psf.clone()
em_p_true = em_true_psf.clone()
em_p_gt = em_gt.clone()

mask = (
        (em_p_true.xyz[:, 0] < 58)
        & (em_p_true.xyz[:, 0] > 8)
        & (em_p_true.xyz[:, 1] < 58)
        & (em_p_true.xyz[:, 1] > 8)
    )
em_p_true = em_p_true[mask]
em_p_true = em_p_true[em_p_true.phot > 300]

em_p_wrong.xyz[:,0] = em_p_wrong.xyz[:,0] + 5.7535/100
em_p_wrong.xyz[:,1] = em_p_wrong.xyz[:,1] + 64.5357/100
path_out = "../../results/" + f"{pframe.stem}_decode_plex_fit_{model_identifier}_wrong_subtract.h5"
em_p_wrong.save(path_out)

em_p_debias.xyz[:,0] = em_p_debias.xyz[:,0] + 5.6270/100
em_p_debias.xyz[:,1] = em_p_debias.xyz[:,1] + 64.4892/100
path_out = "../../results/" + f"{pframe.stem}_decode_plex_fit_{model_identifier}_debias_subtract.h5"
em_p_debias.save(path_out)

em_p_true.xyz[:,0] = em_p_true.xyz[:,0] + 17.8951/100
em_p_true.xyz[:,1] = em_p_true.xyz[:,1] + 56.6878/100
path_out = "../../results/" + f"{pframe.stem}_decode_plex_fit_{model_identifier}_true_subtract.h5"
em_p_true.save(path_out)

prob = 0.6
xyz_sig_lat_nm = 40

# em_p_wrong = em_p_wrong[em_p_wrong.phot > 300]
# em_p_wrong = em_p_wrong[em_p_wrong.xyz_sig_lat_nm < xyz_sig_lat_nm]

# em_p_debias = em_p_debias[em_p_debias.prob > prob]
# em_p_debias = em_p_debias[em_p_debias.xyz_sig_lat_nm < xyz_sig_lat_nm]

group_size = 1
em_p_debias.frame_ix = (em_p_debias.frame_ix // group_size).long()
em_p_gt.frame_ix = (em_p_gt.frame_ix // group_size).long()

d_tp, d_fp, d_fn, d_tp_match = matcher.forward(em_p_debias, em_p_gt)
metrics = evaluator.forward(d_tp, d_fp, d_fn, d_tp_match)
debias_psf_rmse_lat = metrics["rmse_lat"]
debias_psf_rmse_ax = metrics["rmse_ax"]
print((d_tp_match.xyz[:,0]-d_tp.xyz[:,0]*100).mean())
print((d_tp_match.xyz[:,1]-d_tp.xyz[:,1]*100).mean())
print(f"Debias PSF RMSE LAT: {debias_psf_rmse_lat:.2f} nm")

em_p_wrong.frame_ix = (em_p_wrong.frame_ix // group_size).long()
# em_p_wrong.frame_ix = torch.zeros_like(em_p_wrong.frame_ix)
tp, fp, fn, tp_match = matcher.forward(em_p_wrong, em_p_gt)
metrics = evaluator.forward(tp, fp, fn, tp_match)
wrong_psf_rmse_lat = metrics["rmse_lat"]
wrong_psf_rmse_ax = metrics["rmse_ax"]
print((tp_match.xyz[:,0]-tp.xyz[:,0]*100).mean())
print((tp_match.xyz[:,1]-tp.xyz[:,1]*100).mean())
print(f"Wrong PSF RMSE LAT: {wrong_psf_rmse_lat:.2f} nm")

em_p_true.frame_ix = (em_p_true.frame_ix // group_size).long()
t_tp, t_fp, t_fn, t_tp_match = matcher.forward(em_p_true, em_p_gt)
metrics = evaluator.forward(t_tp, t_fp, t_fn, t_tp_match)
true_psf_rmse_lat = metrics["rmse_lat"]
true_psf_rmse_ax = metrics["rmse_ax"]
print((t_tp_match.xyz[:,0]-t_tp.xyz[:,0]*100).mean())
print((t_tp_match.xyz[:,1]-t_tp.xyz[:,1]*100).mean())
print(f"True PSF RMSE LAT: {true_psf_rmse_lat:.2f} nm")

In [ ]:
for dim in [0, 1]:
    fig, ax = plt.subplots(figsize=(5, 4))
    for k, em in {"tp": tp, "matched ref": tp_match}.items():
        values = decode.evaluation.predict_dist.px_pointer_dist(em.xyz_px[:, dim], -0.5, 1.)
        ax.hist(values, bins=torch.linspace(-0.5, 0.5, 20), density=True, label=k, alpha=0.6)
    ax.legend(["DECODE-Plex", "Ground Truth"])
    ax.set_xlabel("subpixel position")
    ax.set_ylabel("density")

    plt.tight_layout()
    # plt.savefig(path_out / f"em_out_wrong_psf_dim_{dim}", dpi=300)

In [ ]:
# rendering
em_p = em_wrong_psf.clone()
# em_p = em_p[em_p.phot > 300]

px_size = em_p.px_size

xextent = (0 * px_size[0], 60 * px_size[0])
yextent = (0 * px_size[0], 60 * px_size[1])
zextent = (-200., 200.)

renderer = decode.renderer.renderer.Renderer2D(
    xextent=xextent,
    yextent=xextent,
    colextent=zextent,
    px_size=10.,
    sigma_blur=10.,
    rel_clip=0.05,
    contrast=2.,
    cmap="turbo",
)

img = renderer.forward(em_p, (em_p.xyz[:, 2]))

fig, ax = plt.subplots(figsize=(8, 8))

im = ax.imshow(img.permute(1, 0, 2))

plt.xticks([])
plt.yticks([])
# plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
# Create an inset_axes for the colorbar next to each subplot
cmap = mpl.cm.turbo
# Adjust the [left, bottom, width, height] values as needed for your layout
cbar_ax = ax.inset_axes([1.02, 0., 0.05, 1.])

# Create a ScalarMappable with the turbo colormap and normalization
norm = mpl.colors.Normalize(vmin=zextent[0], vmax=zextent[1])
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# Add the colorbar to the inset_axes
fig.colorbar(sm, cax=cbar_ax, fraction=0.046, pad=0.04)

# decode.renderer.utils.scalebar(img.shape[1] - 120, img.shape[0] - 20, 100, ax=ax)

plt.tight_layout()
# plt.savefig(path_out / "em_out_wrong_psf_rendering.png", dpi=300)